# Diffusion Noise Models

This page describes the diffusion noise model available in `synference`. In contrast to the empirical noise models discussed in the previous sections, which simply model per-band uncertainties as a function of magnitude using binned interpolation, the diffusion noise model is a generative model that learns to the full latent distribution of noise in the data. This allows it to capture more complex noise properties, such as correlations between bands, and to generate more realistic noise realizations.

This noise model is based on the diffusion probabilistic model framework, which has been successfully applied to a wide range of generative modeling tasks in machine learning. The basic idea is to model the noise as a stochastic process that gradually transforms a simple initial distribution (e.g., Gaussian noise) into the complex noise distribution observed in the data. By learning this transformation, the model can generate realistic noise samples that can be added to synthetic photometry to better match the properties of real observations.

This is implemented as the `ScoreBasedUncertaintyModel`, which learns the conditional uncertainty distribution p(σ|m), where σ is the uncertainty and m is the magnitude. It lives in the standalone [syntillate](https://github.com/synthesizer-project/syntillate) package, which Synference installs as a dependency and re-exports from `synference.noise_models`, so it can be used and serialized alongside Synference's other uncertainty models. The model is trained on a dataset of real photometric measurements, where the input is the magnitude and the target is the corresponding (log-scaled) uncertainty. Once trained, the model can be used to generate noise samples for synthetic photometry by sampling from the learned distribution.

We will demonstate this on the standard COSMOSO2020 'Farmer' catalog (Weaver et al. 2020). If you want to train the noise model yourself, you will have to download the full catalog (~3 GB) from [here](https://cosmos2020.calet.org/).



In [ ]:
from syntillate import ScoreBasedUncertaintyModel

?ScoreBasedUncertaintyModel

### Training the Noise Model

The first part is just loading and cleaning the COSMOS2020 catalog, which is fairly standard. We filter low signal to noise sources which are faint in IRAC Ch. 1. As we are interested in learning the full conditional distribution of noise across all the filters, we also remove sources with missing photometry in any of the filters.

```python

from astropy.table import Table
import numpy as np

table = Table.read('COSMOS2020_FARMER_R1_v2.2_p3.fits', memmap=True)

bands = ['CFHT_u', 'HSC_g', 'HSC_r', 'HSC_i', 'HSC_z',
        'HSC_y', 'UVISTA_Y', 'UVISTA_J', 'UVISTA_H', 
        'UVISTA_Ks', 'IRAC_CH1', 'IRAC_CH2']

snr_filter = table['IRAC_CH1_MAG'] < 26
table = table[snr_filter]

for band in bands:
    table = table[np.isfinite(table[band + '_MAG'])]


We then construct our training dataset, which consists of the magnitudes and uncertainties for each band. 

```python

mag_array = np.array([table[band + '_MAG'] for band in bands]).T
flux_error_array = np.array([table[band + '_FLUXERR'] for band in bands]).T

```

Next, we can train the `ScoreBasedUncertaintyModel` on this dataset. This will learn the conditional distribution of uncertainties given magnitudes for each band. We will leave the model architecture as it's default, which is a simple MLP with 5 layers and a hidden dimension of 256. The training process can take some time, especially if you are using a CPU, so be patient!

You can adjust the number of epochs and batch size as needed --> larger batch sizes will speed up training but require more memory, while more epochs will allow the model to learn better but will take longer.

```python

from syntillate import ScoreBasedUncertaintyModel
noise_model = ScoreBasedUncertaintyModel(filter_names=bands)

# COSMOS2020 FLUXERR columns are in uJy; the model reports sigma back in
# the same units (see noise_model.sigma_units).
noise_model.fit(
    mag_array,
    flux_error_array,
    flux_uncertainty_units='uJy',
    n_epochs=200,
    batch_size=1024,
)

```

### Saving the Model

We can save the trained model to disk for later use - the model can be serialized directly into a custom HDF5 format, and re-loaded later without needing to re-train, or rely on unstable dependencies like `pickle` or `torch.save`.

```python

import h5py
with h5py.File('COSMOS_noise_model.h5', 'w') as f:
    noise_model.serialize_to_hdf5(f.create_group('noise_model'))

```

### Loading the Model

We can load in the saved model from disk, just to prove it works! Here we will switch to our pre-trained model which you can get by running `synference-download`, but feel free to load in your own trained model if you have one.


In [ ]:
import h5py

with h5py.File("COSMOS_noise_model.h5", "r") as f:
    noise_model = ScoreBasedUncertaintyModel._from_hdf5_group(f["noise_model"])

### Validating the Noise Model

### Using the Noise Model in Synference